# OWLv2 base/16 ensemble — DIMER open-vocabulary object detection tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/owlv2-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/owlv2-detection-pipeline/blob/main/tutorials/owlv2_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fowlv2--base--patch16--ensemble-ffcc4d?style=flat)](https://huggingface.co/google/owlv2-base-patch16-ensemble) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fscenic%20(owl__vit)-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/scenic/tree/main/scenic/projects/owl_vit) [![arXiv](https://img.shields.io/badge/arXiv-2306.09683-b31b1b.svg)](https://arxiv.org/abs/2306.09683)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot (open-vocabulary, text-prompted) object detection using the pinned `google/owlv2-base-patch16-ensemble` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/owlv2_detection_pipeline/pipeline.py` at revision `5a8fa2f89a5e`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `cfd3195ba4ea9592eec887ded089f4c08eff231d` (~622 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the image is padded to a square with grey on the bottom and right, resized to 960×960 and split into 60×60 = 3,600 patches by a CLIP ViT-B/16 image tower; each of the caller's phrases is encoded by the CLIP text tower into one query embedding; every image patch proposes one box and is scored against every query by a **sigmoid** of the image–text logit; the box survives when its best query score reaches the threshold and is labelled with that query. The pipeline returns each surviving box in xyxy pixel coordinates of the input image, the phrase it matched, and its score. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried pipeline module adds snapshot verification, prompt and image validation with named ceilings, a fixed output contract and the `box_iou`, `validate_inputs` and `evaluation_report` helpers. The default sample is a synthetic scene drawn in code; the IoU values reported for it are sanity evidence against the boxes you drew, not a benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic scene with known object boxes, validate the image and the prompts into an input manifest through the pipeline's own validation stage, run text-prompted detection through the public API with an explicit caller-owned threshold, read sigmoid scores and the absence of non-maximum suppression correctly, produce an evaluation report that is `sample-sanity` with `box_iou` only when reference boxes exist and `not-measurable` otherwise, exercise an optional BYOD path, and export machine-readable detections plus an annotated image and provenance.

**This notebook does not demonstrate:** instance or semantic segmentation (see the sibling SAM 2 pipeline), tracking, OCR, captioning, closed-set detection with a fixed class list (see the sibling RT-DETR pipeline), image-guided (one-shot) detection, mAP or precision/recall evaluation (which needs a labelled box set), or any training. Prompts are free text limited to 16 CLIP tokens each, so a long phrase is silently truncated; and there is no non-maximum suppression, so one object can surface as several overlapping boxes at a low threshold — the score, not the label, is your only signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate for one image: the repository's model card records 5.3 s to load and 2.9–3.4 s per `detect` on the 640×480 synthetic scene in the Windows venv (Intel Core Ultra 9 275HX) — the cost is the fixed 960×960 ViT-B/16 pass, not the image size. The pinned `torch==2.14.0` install and the 620 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union measures; what a sigmoid score is and why it is not a probability.
- **Data:** the default sample is a deterministic 640×480 scene drawn in code (grey background, a black rectangle, a red disc, a blue triangle) with three prompts naming them, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus your own comma-separated prompt phrases (1–16 distinct phrases, at most 48 characters each; the upstream convention is `a photo of a <thing>`). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/owlv2-base-patch16-ensemble` snapshot (~622 MB in total) at revision `cfd3195ba4ea…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'owlv2-detection-pipeline',
    'repository_revision': '5a8fa2f89a5e1c54a45e9c66506fe872acb0c72f',
    'embedded_module': 'src/owlv2_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/owlv2_detection_pipeline/pipeline.py'],
    'module_sha256': '757a7f0bc8702531836679b25443f9fdb6235c31f7b90d6a47eae6962604ba13',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/owlv2_detection_pipeline/` @ `5a8fa2f89a5e`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/owlv2_detection_pipeline/pipeline.py`

In [ ]:
"""Open-vocabulary (text-prompted) object detection with the pinned ``google/owlv2-base-patch16-ensemble``
checkpoint (OWLv2).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the OWLv2 architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "google/owlv2-base-patch16-ensemble"
MODEL_REVISION = "cfd3195ba4ea9592eec887ded089f4c08eff231d"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "owlv2-base-patch16-ensemble"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Threshold: the value the pinned README's usage example passes to post_process_object_detection
# (threshold=0.1). It gates a sigmoid over the best text query per image patch that is not calibrated;
# the deployment owns tuning it on its own labelled data.
DETECTION_THRESHOLD = 0.1
# The ViT-B/16 image tower sees a 960x960 padded square as 60x60 = 3600 patch tokens, each of which
# is one detection candidate, so no image can yield more than this many boxes.
MAX_DETECTIONS = 3600
# Input ceilings. The processor pads the image to a square with grey (bottom/right) and resizes it to
# 960x960 (preprocessor_config.json), so image cost is bounded; each text query is tokenised by the
# CLIP tokenizer with model_max_length 16 (padded/truncated), so a phrase longer than that is cut.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
MAX_PROMPT_CHARS = 48
MAX_TEXT_TOKENS = 16


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def format_prompts(prompts: Sequence[str]) -> list[str]:
    """Validate a list of phrases and normalise them to the OWLv2 query form: stripped, lower-cased, one
    text query per phrase (the upstream example uses "a photo of a cat"-style queries; the pipeline
    passes the caller's phrases through unchanged apart from case and whitespace)."""
    if isinstance(prompts, str) or not isinstance(prompts, Sequence):
        raise TypeError("prompts must be a list of phrases, not a single string")
    if not 1 <= len(prompts) <= MAX_PROMPTS:
        raise ValueError(f"prompt count {len(prompts)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
    cleaned: list[str] = []
    for phrase in prompts:
        if not isinstance(phrase, str):
            raise TypeError(f"prompt must be str, got {type(phrase).__name__}")
        text = " ".join(phrase.split()).strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("prompt phrases must not be empty")
        if len(text) > MAX_PROMPT_CHARS:
            raise ValueError(
                f"prompt {text[:12]!r}... is {len(text)} chars > MAX_PROMPT_CHARS {MAX_PROMPT_CHARS}"
            )
        cleaned.append(text)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("prompt phrases must be distinct after normalisation")
    return cleaned


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one PIL.Image.Image (any mode, converted to RGB) plus 1..MAX_PROMPTS free-text phrases",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prompts": [1, MAX_PROMPTS],
    "prompt_chars": [1, MAX_PROMPT_CHARS],
    "prompt_tokens_per_query": [1, MAX_TEXT_TOKENS],
    "threshold": [0.0, 1.0],
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB, padded to a square with grey on the bottom/right and resized to 960x960 "
        "(CLIP mean/std); phrases stripped and lower-cased into one CLIP text query each (format_prompts, "
        "16-token limit per query); returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, prompts: Any, threshold: Any) -> tuple[Image.Image, list[str], float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``detect`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    queries = format_prompts(prompts)
    checked = _check_threshold("threshold", threshold)
    return rgb, queries, checked


def validate_inputs(
    image: Image.Image,
    prompts: Sequence[str],
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, queries, checked = _check_inputs(image, prompts, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_prompts": len(prompts),
            }
        ],
        "queries": queries,
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[float]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (phrase -> xyxy reference box) the report carries one ``box_iou``
    entry per reference as sample-sanity geometry evidence; without them the verdict is
    ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    detections = list(result["detections"])
    base = {
        "task": "zero-shot (open-vocabulary, text-prompted) object detection",
        "decision_rule": (
            "each of the 3600 image patches proposes one box labelled with its best-matching text query; "
            "the box survives when the sigmoid of that best image-text logit reaches the threshold; the "
            "score is an uncalibrated sigmoid, not a probability, and is not exclusive across queries"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth boxes were supplied for the evaluated image",
            "needs": (
                "labelled boxes on your own images with a phrase vocabulary matching the prompts, "
                "scored per object with box_iou and aggregated into precision/recall or mean average "
                "precision at a stated IoU threshold; no such labelled set ships with this repository"
            ),
        }
    metrics = []
    for phrase, box in ground_truth_boxes.items():
        ious = [box_iou(det["box"], box) for det in detections]
        best = max(range(len(ious)), key=ious.__getitem__) if ious else None
        metrics.append(
            {
                "id": "box_iou",
                "reference": phrase,
                "value": ious[best] if best is not None else 0.0,
                "matched_label": detections[best]["label"] if best is not None else None,
                "label_matches_reference": (detections[best]["label"] == phrase)
                if best is not None
                else False,
                "estimation": "one reference box per phrase on a single scene, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial sample; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled box set from the deployment domain with a matching phrase vocabulary for any "
            "mean-average-precision or precision/recall claim"
        ),
    }


@dataclass
class Owlv2DetectionPipeline:
    """Text-prompted (open-vocabulary) object detection over the pinned OWLv2 base/16 ensemble checkpoint."""

    _runner: Callable[[Image.Image, list[str], float], list[dict[str, Any]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Owlv2DetectionPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Owlv2ForObjectDetection, Owlv2Processor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Owlv2Processor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = Owlv2ForObjectDetection.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image: Image.Image, queries: list[str], threshold: float) -> list[dict]:
            # One text query per phrase; the CLIP tokenizer pads/truncates each to MAX_TEXT_TOKENS.
            inputs = processor(images=image, text=[queries], return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
            # The pinned processor scales boxes by max(height, width) itself because OWLv2 pads the
            # image to a square before resizing; passing the original size is the documented contract.
            result = processor.post_process_grounded_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]], text_labels=[queries]
            )[0]
            return [
                {"box": [float(v) for v in box.tolist()], "label": str(label), "score": float(score)}
                for box, label, score in zip(
                    result["boxes"], result["text_labels"], result["scores"], strict=True
                )
            ]

        return cls(runner, resolved_device)

    def detect(
        self,
        image: Image.Image,
        prompts: Sequence[str],
        *,
        threshold: float = DETECTION_THRESHOLD,
    ) -> dict[str, Any]:
        """Detect the phrases in `prompts`; boxes are xyxy pixel coordinates in the input image."""
        rgb, queries, checked = _check_inputs(image, prompts, threshold)
        detections = self._runner(rgb, queries, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > MAX_DETECTIONS {MAX_DETECTIONS}"
            )
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4 or det["label"] not in queries:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "queries": queries,
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `9`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `cfd3195ba4ea…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Owlv2DetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "owlv2-base-patch16-ensemble",
  "modelId": "google/owlv2-base-patch16-ensemble",
  "revision": "cfd3195ba4ea9592eec887ded089f4c08eff231d",
  "files": [
    {
      "path": "README.md",
      "bytes": 4838,
      "sha256": "7c7426bc5ec939a42d1f96fb093031b6263400cceac4129ebb941a0c8c11b9b9"
    },
    {
      "path": "added_tokens.json",
      "bytes": 67,
      "sha256": "e5dc0da35d20111e8ff3fdfc03682beca23d5f94ed74331bce81786b2636a24f"
    },
    {
      "path": "config.json",
      "bytes": 414,
      "sha256": "ba9df8c25a4b8461887dd0a93d9252c9cd84697fe8d49a9d8794ce409af9acb2"
    },
    {
      "path": "merges.txt",
      "bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "path": "model.safetensors",
      "bytes": 619918824,
      "sha256": "e1e130b9e404cf91a75ad45644c1da9d7fa5284085eecc864266a6923efb99e7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 425,
      "sha256": "cf3e396635b797ee1a464e1b2836e98748f8edac19e89aaa2c93b55ac15b0064"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 121,
      "sha256": "d6e2b9cf664efbad2d22998b8d3da986abcbeed3e0825ad33605c9401f9cf73e"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 1100,
      "sha256": "b55cda6198e152ded427c8a9b3faf1cccf27a7fa080697a62f6ff143f511f44f"
    },
    {
      "path": "vocab.json",
      "bytes": 1059962,
      "sha256": "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"
    }
  ],
  "totalBytes": 621510370
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Owlv2DetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own reference boxes: a deterministic 640×480 RGB scene is drawn in code — grey background, a black filled rectangle at `[80, 120, 280, 360]`, a red filled disc whose bounding box is `[380, 140, 560, 320]` and a blue filled triangle whose bounding box is `[200, 380, 360, 460]` — and the prompts name them (`a black rectangle`, `a red circle`, `a blue triangle`). This is the same scene and prompt set the repository's smoke run used. The drawn boxes are the reference for the `box_iou` sanity check later; they are not a labelled dataset, so nothing here is a precision/recall measurement. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one image and set `BYOD_PROMPTS` to the phrases you want found — no reference boxes exist for it, so the evaluation report will be `not-measurable`.

The detection threshold is a **caller-owned request parameter**, not a pipeline constant: a patch's box survives when the sigmoid of its best image–text logit reaches it. The package default (`DETECTION_THRESHOLD = 0.1`) follows the pinned README's usage example, not a calibration; it is exposed here as a form parameter and passed explicitly on every call. Nothing is validated in this cell — the next section hands the image, the prompts and the threshold to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the prompts, the threshold, and the drawn reference boxes.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PROMPTS = 'a photo of a cat, a photo of a remote control'  # @param {type:"string"}
threshold = 0.1  # @param {type:"number"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    prompts = [phrase.strip() for phrase in BYOD_PROMPTS.split(',') if phrase.strip()]
    drawn_boxes = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable.
    image = Image.new('RGB', (640, 480), (128, 128, 128))
    draw = ImageDraw.Draw(image)
    drawn_boxes = {'a black rectangle': [80.0, 120.0, 280.0, 360.0], 'a red circle': [380.0, 140.0, 560.0, 320.0], 'a blue triangle': [200.0, 380.0, 360.0, 460.0]}
    draw.rectangle(drawn_boxes['a black rectangle'], fill=(30, 30, 30))
    draw.ellipse(drawn_boxes['a red circle'], fill=(220, 30, 30))
    draw.polygon([(280, 380), (200, 460), (360, 460)], fill=(30, 60, 220))
    prompts = list(drawn_boxes)
    image_name = 'synthetic_scene_640x480.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'prompts': prompts, 'threshold': threshold, 'drawn_boxes': drawn_boxes})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `detect` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, 1..`MAX_PROMPTS` distinct phrases of at most `MAX_PROMPT_CHARS` characters each, and a threshold in `[0, 1]` — canonicalises the phrases through the package's `format_prompts` (whitespace-collapsed, lower-cased, trailing period removed, one text query each) and returns an **input manifest** naming the schema and ceilings (including the 3,600-patch ceiling on detections and the 16-token CLIP limit per query), the input's observed mode and size, the queries actually sent to the tokenizer, the threshold, and the verdict. The manifest is written to `outputs/owlv2_detection_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately over-long phrase and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB, padded to a square and resized to 960×960 by the processor; boxes are mapped back to input pixels, and nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PROMPTS': MAX_PROMPTS, 'MAX_PROMPT_CHARS': MAX_PROMPT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_DETECTIONS': MAX_DETECTIONS, 'DETECTION_THRESHOLD': DETECTION_THRESHOLD}})
input_manifest = validate_inputs(image, prompts, threshold=threshold, names=[image_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(image, ['x' * (MAX_PROMPT_CHARS + 1)])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-long-prompt-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/owlv2_detection_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Detect and read the scores correctly

`detect` returns a dict with `detections` — a list of `{box, label, score}` **ordered by descending score**, `box` in xyxy pixel coordinates of the input, `label` the matched query text — plus `queries`, the threshold used, `width`, `height` and the model identity. At most 3,600 boxes can ever be returned (one per image patch). Each `score` is a **sigmoid of the best image–text logit, not a calibrated probability**: it was never fitted to the frequency with which a box is correct, so 0.9 does not mean "90 % likely", and the scores of different queries for one patch are independent. The threshold you passed is the only decision rule; the pipeline ships 0.1 as a default (the README example's value), not as a calibration, and the caller owns it per deployment — raise it when false boxes cost more than missed ones, lower it for recall. **There is no non-maximum suppression**: neighbouring patches can propose overlapping boxes for the same object, and at a low threshold the same object appears more than once. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place and reorder near-ties. As recorded in the model card, the repository's CPU smoke on this same scene at the default threshold returned exactly three boxes — `a red circle` 0.917, `a blue triangle` 0.810, `a black rectangle` 0.426 — within about 4 px of the drawn ones, and five boxes at threshold 0.05; that is one observation, not a calibration point.

In [ ]:
import time

t0 = time.time()
result = pipe.detect(image, prompts, threshold=threshold)
elapsed = time.time() - t0
print({'n_detections': len(result['detections']), 'queries': result['queries'], 'threshold': result['threshold'], 'device': pipe.device, 'seconds': round(elapsed, 2)})
for rank, det in enumerate(result['detections'], start=1):
    print(f"{rank:>2}. score {det['score']:.4f}  label {det['label']!r}  box {[round(v, 1) for v in det['box']]}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No detection metric is reported by default: mean average precision needs a labelled box set with a matching vocabulary, and this repository ships none. The repository's only metric helper is `box_iou(a, b)` (intersection-over-union of two xyxy boxes), the building block a caller would use to compute mAP on their own labelled data; when reference boxes are supplied the report carries one `box_iou` entry per reference — its value, which detection matched it best and whether that detection's label agrees — with the verdict `sample-sanity`. On the synthetic path those references are shapes **you drew yourself**, so a high IoU proves only that the input contract, query formatting, forward pass and coordinate mapping (including the square padding) round-trip. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable: labelled boxes on your own images with a phrase vocabulary matching the prompts. The report is written to `outputs/owlv2_detection_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, drawn_boxes, sample_kind=sample_kind)
with open('outputs/owlv2_detection_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No reference boxes exist for this input, so box_iou is not computed; inspect the annotated PNG instead.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (score-ordered detections with boxes and labels, the queries, the threshold), the evaluation report, the input manifest, the sample identity, digest and drawn boxes, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The detections are also written as CSV with explicit `image`, `rank`, `label`, `score`, `x0`, `y0`, `x1`, `y1` columns so score ordering survives downstream use, and an annotated PNG draws every returned box for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv

annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for det in result['detections']:
    draw.rectangle(det['box'], outline=(0, 255, 0), width=2)
    draw.text((det['box'][0] + 2, det['box'][1] + 2), f"{det['label']} {det['score']:.2f}", fill=(0, 255, 0))
annotated.save('outputs/owlv2_detection_annotated.png')
payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'prompts': prompts, 'drawn_boxes': drawn_boxes},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/owlv2_detection_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/owlv2_detection_detections.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for rank, det in enumerate(result['detections'], start=1):
        writer.writerow([image_name, rank, det['label'], f"{det['score']:.6f}", *[f"{v:.2f}" for v in det['box']]])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The boxes are matched to free-text phrases: the label tells you which query the box scored best against, not that the object is really there, and the sigmoid score is uncalibrated. The threshold is a request parameter you own; the default is the README usage example, not a tuned operating point. On the synthetic scene the `box_iou` values in the evaluation report compare detections to shapes you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, query formatting, forward pass and coordinate mapping work; they say nothing about photographs, small or occluded objects, crowded scenes, or vocabulary the model has never seen, and a BYOD result is a single-image observation with the verdict `not-measurable`. Prompts longer than 16 CLIP tokens are truncated silently, the image is padded to a square so a wide or tall image is encoded at a lower effective resolution, and without non-maximum suppression one object can surface as several boxes at a low threshold — the smoke run's icon scene showed duplicate `clock` and `stop sign` boxes at 0.11 and 0.106 alongside the real ones. Empty input is handled better than a closed-set detector's: a blank 4096×4096 image returned no boxes at the default threshold. The pipeline provides no segmentation, tracking, OCR, captioning, mAP evaluation, or training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** lower `threshold` to 0.05 and count the duplicate boxes (the smoke run found five for three shapes); rephrase the prompts in the upstream form (`a photo of a red circle`) and compare scores; add a prompt for something that is not in the scene (`a green star`) and see whether anything surfaces; enable `USE_BYOD` with a photograph, hand-label a few objects and pass them to `evaluation_report` to see the verdict switch to `sample-sanity` — the first step towards a real precision/recall number.

## References

- Repository README: https://github.com/kurtvalcorza/owlv2-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/owlv2-detection-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/owlv2-detection-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/owlv2-base-patch16-ensemble
- Upstream code (Scenic, OWL-ViT project): https://github.com/google-research/scenic/tree/main/scenic/projects/owl_vit
- Scaling Open-Vocabulary Object Detection (Minderer, Gritsenko, Houlsby, 2023): https://arxiv.org/abs/2306.09683
- Simple Open-Vocabulary Object Detection with Vision Transformers (Minderer et al., 2022): https://arxiv.org/abs/2205.06230